# Reward model robustness on HH-RLHF

Train a small reward model, then probe what it actually learned: length bias,
sycophancy, position sensitivity, and behaviour under distribution shift.

**Runtime on a T4** (free Colab tier is enough):

| step | time |
|---|---|
| setup + data | ~3 min |
| day-0 baselines (no model needed) | ~1 min |
| train `distilroberta-base`, 1 epoch on helpful-base | ~35-50 min |
| full probe suite + report | ~8 min |
| control arm (length-balanced) | ~25 min |
| shift arm (trained on harmless-base) | ~35-50 min |

Read `PREREGISTRATION.md` before looking at any output. It states what was predicted
and what would falsify each prediction; the last cell checks the run against it.


## 1. Setup


In [ ]:
import subprocess, sys, os
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'],
                     capture_output=True, text=True).stdout or 'NO GPU - switch runtime to T4')


In [ ]:
# Point this at the repo. Either upload rm-robustness.tar.gz and untar, or git clone.
REPO = '/content/rm-robustness'
if not os.path.isdir(REPO):
    tars = [f for f in os.listdir('/content') if f.endswith('.tar.gz')]
    if tars:
        !mkdir -p {REPO} && tar xzf /content/{tars[0]} -C /content
    else:
        raise SystemExit('Upload rm-robustness.tar.gz to /content, or git clone the repo to ' + REPO)
os.chdir(REPO)
print(os.getcwd()); print(sorted(os.listdir('.')))


In [ ]:
!pip install -q -r requirements.txt
# deberta-v3 backbones additionally need sentencepiece; distilroberta-base does not.
# !pip install -q sentencepiece


In [ ]:
!python scripts/fetch_data.py


In [ ]:
!python -m pytest tests -q    # 36 tests, ~15s. If these fail, stop here.


## 2. The bar, before any model exists

Every number below is available from the dataset alone. A reward model that does not
beat these is a length detector with extra steps.


In [ ]:
!python scripts/day0_baselines.py


## 3. Train the main arm

`distilroberta-base` on `helpful-base`. Swap in `microsoft/deberta-v3-small` for a
stronger model at ~1.6x the time (and `pip install sentencepiece` above).


In [ ]:
!python -m rmrobust.cli train \
  --backbone distilroberta-base \
  --train-sources helpful-base \
  --max-length 512 --batch-size 8 --grad-accum 2 --lr 1e-5 --epochs 1 \
  --eval-every 300 --seed 0 \
  --out runs/base


In [ ]:
import json
s = json.load(open('runs/base/train_summary.json'))
print('best val acc', s['best_val'], 'over', s['total_steps'], 'steps',
      f"in {s['wall_seconds']/60:.1f} min")


## 4. Probe it

Scores the four HH test splits, runs all four probe families, writes
`runs/base/results.json`, `report.md` and `figures/`.


In [ ]:
!python -m rmrobust.cli probe \
  --checkpoint runs/base/best \
  --train-sources helpful-base --reference-source helpful-base \
  --batch-size 32 --n-boot 2000 --quiet \
  --out runs/base


In [ ]:
from IPython.display import Markdown, display
display(Markdown(open('runs/base/report.md').read()))


In [ ]:
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('runs/base/figures/*.png')):
    print(f); display(Image(f))


## 5. The first check the pre-registration demands

If the reward model is not clearly ahead of the calibrated length baseline on a
subset, nothing measured on that subset is interpretable.


In [ ]:
import json
r = json.load(open('runs/base/results.json'))
print(f"{'subset':30s} {'RM':>7s} {'length':>7s} {'gap':>7s}")
for src, d in sorted(r['length']['by_source'].items()):
    if d.get('insufficient_data'): continue
    a = d['comparative']['acc_rm']['value']
    b = d['comparative']['acc_length_logistic_oof']['value']
    flag = '' if a - b > 0.02 else '   <-- too close to call'
    print(f'{src:30s} {a:7.3f} {b:7.3f} {a-b:+7.3f}{flag}')


## 6. Control arm: train with length carrying no signal

`--length-balanced` resamples so that the sign and magnitude of the length difference
are uncorrelated with the preference label. If accuracy collapses to chance, the model
had nothing but length; if it holds, there is real signal underneath. This is the
cleanest single check on the headline decomposition.


In [ ]:
!python -m rmrobust.cli all \
  --backbone distilroberta-base \
  --train-sources helpful-base --reference-source helpful-base \
  --max-length 512 --batch-size 8 --grad-accum 2 --lr 1e-5 --epochs 1 \
  --eval-every 300 --seed 0 --length-balanced \
  --batch-size 8 --n-boot 2000 --quiet \
  --out runs/balanced


## 7. Shift arm: train on the subset that wants the opposite

harmless-base prefers the *shorter* response. Training here and probing everywhere
tests whether the learned length coefficient is domain-conditional (it should not be).


In [ ]:
!python -m rmrobust.cli all \
  --backbone distilroberta-base \
  --train-sources harmless-base --reference-source harmless-base \
  --max-length 512 --batch-size 8 --grad-accum 2 --lr 1e-5 --epochs 1 \
  --eval-every 300 --seed 0 \
  --n-boot 2000 --quiet \
  --out runs/harmless


## 8. Compare the arms


In [ ]:
import json, os
rows = []
for name in ('base','balanced','harmless'):
    p = f'runs/{name}/results.json'
    if not os.path.exists(p): continue
    r = json.load(open(p))
    for src, d in sorted(r['length']['by_source'].items()):
        if d.get('insufficient_data'): continue
        lef = d['decomposition']['length_explained_fraction']
        rows.append({
            'arm': name, 'eval_subset': src,
            'acc': round(d['comparative']['acc_rm']['value'], 3),
            'acc_length_only': round(d['comparative']['acc_length_logistic_oof']['value'], 3),
            'length_share': (round(lef['value'], 3) if lef.get('reliable', True) else None),
            'reward_per_log_len_sd': round(
                r['shift']['cross_source']['by_source'][src]['length']['reward_per_log_length_in_sd'], 3)
                if 'shift' in r and src in r['shift']['cross_source']['by_source'] else None,
        })
import pandas as pd
df = pd.DataFrame(rows)
display(df.pivot(index='eval_subset', columns='arm', values=['acc','length_share','reward_per_log_len_sd']))
df.to_csv('runs/arm_comparison.csv', index=False)


## 9. Score the run against the pre-registration

Fill this in by hand against `PREREGISTRATION.md`. A falsified prediction is a result,
not a failure — write down which mechanism died and what replaced it.

| # | prediction | outcome | note |
|---|---|---|---|
| P1 | acc 0.65-0.71, length share 0.40-0.75 | | |
| P2 | length share negative on harmless-base | | |
| P3 | length coefficient same sign on all four | | |
| P4 | sycophancy net of placebo in 0.05-0.20 sd | | |
| P5 | flattery positive, capitulation CI covers 0 | | |
| P6 | swap: |effect| 0.05-0.25 sd, signed CI covers 0 | | |
| P7 | 32-token prefix recovers >=60% on helpful-online | | |
| P8 | constraint gap retains 0.3-0.8 at depth 3 | | |
| P9 | lowercase shifts >0.05 sd, flips 2-12% | | |
| P10 | OOD ECE >= 2x in-domain, mean shift >= 0.3 sd | | |
| P11 | length-balanced arm scores 0.55-0.62 | | |
